# rdflib-neo4j — Feature Tour

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/neo4j-labs/rdflib-neo4j/blob/main/examples/rdflib_neo4j_demo.ipynb)

This notebook gives a hands-on tour of `rdflib-neo4j` — the RDFLib store backend
for Neo4j. It covers:

| # | Feature | Config option |
|---|---------|---------------|
| 1 | Basic import via `Graph.parse()` | — |
| 2 | URI naming strategies | `handle_vocab_uri_strategy` |
| 3 | XSD datatype coercion | automatic |
| 4 | `handleRDFTypes` — how `rdf:type` is stored | `handle_rdf_types_strategy` |
| 5 | Blank node → `bnode://` URI mapping | automatic |
| 6 | Preview / dry-run mode | `preview=True` |
| 7 | `graph.remove()` — triple deletion | — |
| 8 | `graph.triples()` / iteration / serialization | — |
| 9 | Multi-value properties | `handle_multival_strategy` |
|10 | Node cache — performance tuning | `cache_size` |

### The story

We manage an RDF knowledge graph for a small research lab: people, publications,
and affiliations. We'll import, inspect, modify and query it step by step.

## 0. Setup

In [ ]:
%pip install -q rdflib-neo4j neo4j python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()  # reads NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE from .env

URI      = os.environ["NEO4J_URI"]
USER     = os.environ["NEO4J_USERNAME"]
PASSWORD = os.environ["NEO4J_PASSWORD"]
DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f"Connected to {URI} / {DATABASE}")

In [ ]:
def cypher(q, **params):
    records, _, _ = driver.execute_query(q, params, database_=DATABASE)
    return records

def reset_db():
    cypher("MATCH (n) DETACH DELETE n")
    # Note: Neo4jStore.open(config, create=True) recreates the constraint/index automatically.

## 1. Basic Import

`Graph(store=Neo4jStore(config))` gives you a standard rdflib graph that
transparently persists every triple to Neo4j. `rdf:type` becomes a label;
literal predicates become properties; URI predicates become relationships.

In [ ]:
from rdflib import Graph
from rdflib_neo4j import Neo4jStoreConfig, Neo4jStore, HANDLE_VOCAB_URI_STRATEGY

TURTLE = """
@prefix schema: <https://schema.org/> .
@prefix ex:     <http://lab.example.org/> .
@prefix xsd:    <http://www.w3.org/2001/XMLSchema#> .

ex:alice a schema:Person ;
    schema:name        "Alice Chen" ;
    schema:email       "alice@lab.example.org" ;
    schema:birthDate   "1990-03-15"^^xsd:date ;
    schema:alumniOf    ex:mit ;
    schema:memberOf    ex:lab .

ex:bob a schema:Person ;
    schema:name        "Bob Smith" ;
    schema:email       "bob@lab.example.org" ;
    schema:birthDate   "1985-07-22"^^xsd:date ;
    schema:memberOf    ex:lab .

ex:lab a schema:Organization ;
    schema:name        "Graph Research Lab" ;
    schema:foundingYear 2018 .

ex:mit a schema:Organization ;
    schema:name "MIT" .

ex:paper1 a schema:ScholarlyArticle ;
    schema:name   "Graph Neural Networks for RDF" ;
    schema:author ex:alice ;
    schema:author ex:bob ;
    schema:datePublished "2023-01-10"^^xsd:date ;
    schema:citation 42 .
"""

reset_db()

auth_data = {"uri": URI, "database": DATABASE, "user": USER, "pwd": PASSWORD}
config = Neo4jStoreConfig(
    auth_data=auth_data,
    custom_prefixes={"schema": "https://schema.org/"},
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    batching=False,
)
g = Graph(store=Neo4jStore(config=config))
g.open(config, create=True)
g.parse(data=TURTLE, format="ttl")
g.close(True)

for r in cypher("MATCH (n:Resource) RETURN labels(n) AS lbl, n.uri AS uri ORDER BY uri"):
    print(f"{str(r['lbl']):30}  {r['uri']}")

## 2. URI Naming Strategies

`handle_vocab_uri_strategy` controls how RDF predicate and type URIs map to
Neo4j property and label names.

| Strategy | `schema:name` becomes | `schema:Person` becomes |
|---|---|---|
| `IGNORE` | `name` | `Person` |
| `SHORTEN` | `schema__name` | `schema__Person` |
| `SHORTEN_STRICT` | `schema__name` (or error if prefix unknown) | `schema__Person` |
| `KEEP` | `https://schema.org/name` | `https://schema.org/Person` |

In [ ]:
# See what names are stored with IGNORE (current import)
for r in cypher("MATCH (p:Person) RETURN keys(p) AS props LIMIT 1"):
    print("Properties with IGNORE strategy:", r["props"])

In [ ]:
# Preview what SHORTEN would produce (no DB write)
config_shorten = Neo4jStoreConfig(
    custom_prefixes={"schema": "https://schema.org/"},
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.SHORTEN,
    preview=True,
)
g_preview = Graph(store=Neo4jStore(config=config_shorten))
g_preview.parse(data="""
    @prefix schema: <https://schema.org/> .
    @prefix ex: <http://lab.example.org/> .
    ex:test a schema:Person ; schema:name \"Test\" .
""", format="ttl")

for q, p in config_shorten.get_preview_results()[:2]:
    print(q[:100])

## 3. XSD Datatype Coercion

Typed RDF literals are automatically coerced to native Neo4j types:

| XSD type | Neo4j type |
|---|---|
| `xsd:integer`, `xsd:long` | Integer |
| `xsd:decimal`, `xsd:float`, `xsd:double` | Float |
| `xsd:boolean` | Boolean |
| `xsd:date` | Date |
| `xsd:dateTime` | DateTime |
| `xsd:string`, `xsd:anyURI` | String |

In [ ]:
for r in cypher(
    "MATCH (p:Person) RETURN p.name AS name, p.birthDate AS bd, "
    "apoc.meta.type(p.birthDate) AS bd_type ORDER BY name"
):
    print(f"{r['name']:15}  birthDate={r['bd']}  neo4j_type={r['bd_type']}")

In [ ]:
# citation count is stored as an integer — use it in arithmetic
for r in cypher(
    "MATCH (a:ScholarlyArticle) RETURN a.name AS title, a.citation AS cites, "
    "a.citation * 10 AS impact_score"
):
    print(r["title"], "— citations:", r["cites"], "→ impact:", r["impact_score"])

## 4. handleRDFTypes — How `rdf:type` Is Stored

By default, `rdf:type` becomes a Neo4j **label** only. You can also store it
as a **relationship** (for queries that need to traverse the type hierarchy)
or as **both**.

In [ ]:
from rdflib_neo4j.config.const import HandleRDFTypesStrategy

SAMPLE = """
@prefix schema: <https://schema.org/> .
@prefix ex:     <http://lab.example.org/> .
ex:demo a schema:Person ; schema:name "Demo" .
"""

for strategy in [HandleRDFTypesStrategy.LABELS,
                 HandleRDFTypesStrategy.NODES,
                 HandleRDFTypesStrategy.LABELS_AND_NODES]:
    cfg = Neo4jStoreConfig(
        custom_prefixes={"schema": "https://schema.org/"},
        handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
        handle_rdf_types_strategy=strategy,
        preview=True,
    )
    g_tmp = Graph(store=Neo4jStore(config=cfg))
    g_tmp.parse(data=SAMPLE, format="ttl")
    print(f"\n── Strategy: {strategy.name} ──")
    for q, _ in cfg.get_preview_results():
        if "Person" in q or "rdf__type" in q or "type" in q.lower():
            print(" ", q[:120])

## 5. Blank Nodes → `bnode://` URIs

RDF blank nodes (anonymous resources) have no URI. rdflib-neo4j maps them to
synthetic `bnode://<id>` URIs so they can be stored as regular `Resource` nodes
and queried back.

In [ ]:
from rdflib import Graph, BNode, URIRef, Literal, RDF
from rdflib import Namespace

SCHEMA = Namespace("https://schema.org/")
EX     = Namespace("http://lab.example.org/")

g_bn = Graph(store=Neo4jStore(config=config))
g_bn.open(config, create=False)

# Add an address as a blank node
addr = BNode()
g_bn.add((EX.alice, SCHEMA.address, addr))
g_bn.add((addr, SCHEMA.streetAddress, Literal("1 Infinite Loop")))
g_bn.add((addr, SCHEMA.addressLocality, Literal("Cambridge")))
g_bn.close(True)

# The blank node is stored with a bnode:// URI
for r in cypher(
    "MATCH (p:Person {name: 'Alice Chen'})-[:address]->(addr) "
    "RETURN addr.uri AS bnode_uri, addr.streetAddress AS street, addr.addressLocality AS city"
):
    print("BNode URI:", r["bnode_uri"])
    print("Street:   ", r["street"])
    print("City:     ", r["city"])

## 6. Preview / Dry-Run Mode

`preview=True` captures all Cypher queries that *would* be sent to Neo4j —
without executing them. Ideal for testing pipelines or auditing without a DB.

In [ ]:
preview_config = Neo4jStoreConfig(
    custom_prefixes={"schema": "https://schema.org/"},
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    preview=True,
)

g_dry = Graph(store=Neo4jStore(config=preview_config))
g_dry.parse(data="""
    @prefix schema: <https://schema.org/> .
    @prefix ex: <http://lab.example.org/> .
    ex:carol a schema:Person ; schema:name \"Carol\" ; schema:email \"carol@lab.org\" .
""", format="ttl")

results = preview_config.get_preview_results()
print(f"Captured {len(results)} queries (no data written):\n")
for i, (q, p) in enumerate(results, 1):
    print(f"[{i}] {q}")
    if p:
        print(f"    params: {p}")
    print()

preview_config.clear_preview_results()

## 7. Removing Triples

`graph.remove((s, p, o))` maps to the appropriate Cypher operation:
- Literal object → `REMOVE n.property` 
- `rdf:type` predicate → `REMOVE n:Label`
- URI/BNode object → `DELETE relationship`
- Wildcards (`None`) → remove all matching triples

In [ ]:
g_edit = Graph(store=Neo4jStore(config=config))
g_edit.open(config, create=False)

print("Before removal:")
for r in cypher("MATCH (p:Person {name: 'Alice Chen'}) RETURN labels(p) AS l, keys(p) AS k"):
    print("  labels:", r["l"])
    print("  props: ", r["k"])

# Remove the email property
g_edit.remove((EX.alice, SCHEMA.email, None))

# Remove the alumniOf relationship
g_edit.remove((EX.alice, SCHEMA.alumniOf, EX.mit))

g_edit.close(True)

print("\nAfter removal:")
for r in cypher("MATCH (p:Person {name: 'Alice Chen'}) RETURN keys(p) AS k"):
    print("  props: ", r["k"])  # email gone

# alumniOf relationship gone
rels = cypher("MATCH (p:Person {name: 'Alice Chen'})-[r:alumniOf]->() RETURN count(r) AS c")
print("  alumniOf rels:", rels[0]["c"])  # should be 0

## 8. Graph Iteration — `triples()`, `__len__()`, `__iter__()`

rdflib-neo4j implements the full rdflib Store interface:
- `len(g)` — count of triples
- `for s, p, o in g` — iterate all triples
- `g.triples((s, p, o))` — pattern-matched iteration
- `g.serialize(format="turtle")` — round-trip to RDF

In [ ]:
g_read = Graph(store=Neo4jStore(config=config))
g_read.open(config, create=False)

print(f"Total triples: {len(g_read)}")

# Pattern match: all rdf:type triples
print("\nTypes in the graph:")
from rdflib import RDF
for s, p, o in g_read.triples((None, RDF.type, None)):
    print(f"  {s.split('/')[-1]:15} a  {o.split('/')[-1]}")

g_read.close()

In [ ]:
# Serialize back to Turtle — full round-trip
g_ser = Graph(store=Neo4jStore(config=config))
g_ser.open(config, create=False)
ttl = g_ser.serialize(format="turtle")
g_ser.close()

# Show first 30 lines
for line in ttl.splitlines()[:30]:
    print(line)

## 9. Multi-Value Properties

By default, writing the same predicate twice overwrites the previous value (`OVERWRITE`).
Use `ARRAY` strategy + `multival_props_names` to accumulate values as a Neo4j list.

In [ ]:
from rdflib_neo4j.config.const import HANDLE_MULTIVAL_STRATEGY

# Re-import paper with both authors stored as an array
multi_config = Neo4jStoreConfig(
    auth_data=auth_data,
    custom_prefixes={"schema": "https://schema.org/"},
    handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
    handle_multival_strategy=HANDLE_MULTIVAL_STRATEGY.ARRAY,
    batching=False,
)

PAPER_TURTLE = """
@prefix schema: <https://schema.org/> .
@prefix ex:     <http://lab.example.org/> .
ex:paper2 a schema:ScholarlyArticle ;
    schema:name   "Multi-Author Paper" ;
    schema:keyword "graphs" ;
    schema:keyword "RDF" ;
    schema:keyword "Neo4j" .
"""

g_multi = Graph(store=Neo4jStore(config=multi_config))
g_multi.open(multi_config, create=False)
g_multi.parse(data=PAPER_TURTLE, format="ttl")
g_multi.close(True)

for r in cypher("MATCH (a:ScholarlyArticle {name: 'Multi-Author Paper'}) RETURN a.keyword AS kw"):
    print("keywords (array):", r["kw"])

## 10. Node Cache — Performance Tuning

By default, every triple write issues a `MERGE` query to Neo4j — even if the
subject node was just created. The in-process node cache skips redundant MERGEs
for nodes already seen in the current session.

Use `cache_size` in `Neo4jStoreConfig` to control memory usage (0 = disabled).

In [ ]:
import time

LARGE_TURTLE = "\n".join([
    f"""<http://lab.example.org/person{i}> a <https://schema.org/Person> ;
    <https://schema.org/name> \"Person {i}\" ;
    <https://schema.org/age>  {20 + i % 50} ."""
    for i in range(200)
])

for cache_size in [0, 500]:
    reset_db()
    cfg = Neo4jStoreConfig(
        auth_data=auth_data,
        custom_prefixes={"schema": "https://schema.org/"},
        handle_vocab_uri_strategy=HANDLE_VOCAB_URI_STRATEGY.IGNORE,
        cache_size=cache_size,
        batching=False,
    )
    g_perf = Graph(store=Neo4jStore(config=cfg))
    g_perf.open(cfg, create=True)  # create=True recreates the constraint
    t0 = time.perf_counter()
    g_perf.parse(data=LARGE_TURTLE, format="ttl")
    g_perf.close(True)
    elapsed = time.perf_counter() - t0
    label = "no cache" if cache_size == 0 else f"cache_size={cache_size}"
    print(f"{label:20}  {elapsed:.2f}s")

## Appendix — What's in the Other Notebooks

| Notebook | PR | Topic |
|---|---|---|
| [`cypher_dsl_demo.ipynb`](cypher_dsl_demo.ipynb) | #81 | Typed Cypher 25 DSL — `CypherQuery` builder |
| [`sparql_transpiler_demo.ipynb`](sparql_transpiler_demo.ipynb) | #83 | SPARQL → Cypher transpiler |
| [`named_graphs_demo.ipynb`](named_graphs_demo.ipynb) | #79 | Named graph / quad support |

In [ ]:
driver.close()